# Trabajo Práctico de Laboratorio N° 2

## Analizador Sintáctico Descendente Predictivo por Tabla

---
# 1. Verificación y modificación de la gramática

## 1.1. Gramática original

La gramática original de Mini-C proporcionada en el enunciado es:

```text
Program → DeclList

DeclList → DeclList Decl
         | Decl

Decl → FuncDecl
     | VarDecl

VarDecl → Type id ;
        | Type id = Expr ;

FuncDecl → Type id ( ParamList ) Block
         | Type id ( ) Block

Type → int | float | bool | void | string

ParamList → ParamList , Param
          | Param

Param → Type id

Block → { StmtList }

StmtList → StmtList Stmt
         | Stmt
         | λ

Stmt → VarDecl
     | id = Expr ;
     | id ( ArgList ) ;
     | id ( ) ;
     | IfStmt
     | WhileStmt
     | ForStmt
     | return Expr ;
     | return ;
     | print ( Expr ) ;
     | read ( id ) ;

IfStmt → if ( Expr ) Block else Block
       | if ( Expr ) Block

WhileStmt → while ( Expr ) Block

ForStmt → for ( id = Expr ; Expr ; id = Expr ) Block

Expr → Expr || RelExpr
     | Expr && RelExpr
     | RelExpr

RelExpr → AddExpr RelOp AddExpr
        | AddExpr

RelOp → == | != | < | > | <= | >=

AddExpr → AddExpr + Term
        | AddExpr - Term
        | Term

Term → Term * Unary
     | Term / Unary
     | Unary

Unary → ! Unary
      | - Unary
      | Factor

Factor → ( Expr )
        | id ( ArgList )
        | id ( )
        | id
        | num
        | str
        | true
        | false

ArgList → ArgList , Expr
        | Expr
```
### (El símbolo distinguido de la gramática es: Program)

## 1.2. Eliminación de recursividad izquierda

Se identificaron las siguientes producciones con recursividad izquierda directa:

* DeclList
* ParamList
* StmtList
* Expr
* AddExpr
* Term
* ArgList

Se aplicó el procedimiento de eliminación de recursividad izquierda, obteniendo:
```text
-> DeclList
DeclList → Decl DeclList'
DeclList' → Decl DeclList'
          | λ

->ParamList
ParamList → Param ParamList'
ParamList' → , Param ParamList'
           | λ

->StmtList
StmtList → Stmt StmtList'
StmtList' → Stmt StmtList'
          | λ

->Expr
Expr → RelExpr Expr'
Expr' → || RelExpr Expr'
      | && RelExpr Expr'
      | λ

->AddExpr
AddExpr → Term AddExpr'
AddExpr' → + Term AddExpr'
         | - Term AddExpr'
         | λ

->Term
Term → Unary Term'
Term' → * Unary Term'
      | / Unary Term'
      | λ

->ArgList
ArgList → Expr ArgList'
ArgList' → , Expr ArgList'
         | λ
```
## 1.3. Factorizacion
Luego de eliminar la recursividad izquierda, se realizó la factorización de las producciones que presentan prefijos comunes.
```text
->VarDecl
VarDecl → Type id A
A → ;
  | = Expr ;

B → ParamList ) Block
  | ) Block

->Decl
Decl → Type id C
C → ( B
  | A

->Stmt
Stmt → VarDecl
     | id I
     | IfStmt
     | WhileStmt
     | ForStmt
     | return J
     | print ( Expr ) ;
     | read ( id ) ;
I → = Expr ;
  | ( K
K → ArgList ) ;
  | ) ;
J → Expr ;
  | ;

->IfStmt
  IfStmt → if ( Expr ) Block D
D → else Block
  | λ

->RelExpr
RelExpr → AddExpr E
E → RelOp AddExpr
  | λ

->RelOp
RelOp → ==
      | !=
      | < F
      | > F
F → =
  | λ

->Factor
Factor → ( Expr )
       | id G
       | num
       | str
       | true
       | false
G → ( H
  | λ
H → ArgList )
  | )
```
# 2. Gramática modificada
Despues de eliminar la recursividad izquierda y realizar las factorizaciones necesarias, la gramática utilizada para el análisis sintáctico queda:
```text
Program → DeclList

DeclList → Decl DeclList'

DeclList' → Decl DeclList'
          | λ

Decl → Type id C

C → ( B
  | A

A → ;
  | = Expr ;

VarDecl → Type id A

B → ParamList ) Block
  | ) Block

Type → int | float | bool | void | string

ParamList → Param ParamList'

ParamList' → , Param ParamList'
           | λ

Param → Type id

Block → { StmtList }

StmtList → Stmt StmtList'

StmtList' → Stmt StmtList'
          | λ

Stmt → VarDecl
     | id I
     | IfStmt
     | WhileStmt
     | ForStmt
     | return J
     | print ( Expr ) ;
     | read ( id ) ;

I → = Expr ;
  | ( K

K → ArgList ) ;
  | ) ;

J → Expr ;
  | ;

IfStmt → if ( Expr ) Block D

D → else Block
  | λ

WhileStmt → while ( Expr ) Block

ForStmt → for ( id = Expr ; Expr ; id = Expr ) Block

Expr → RelExpr Expr'

Expr' → || RelExpr Expr'
      | && RelExpr Expr'
      | λ

RelExpr → AddExpr E

E → RelOp AddExpr
  | λ

RelOp → ==
      | !=
      | < F
      | > F

F → =
  | λ

AddExpr → Term AddExpr'

AddExpr' → + Term AddExpr'
         | - Term AddExpr'
         | λ

Term → Unary Term'

Term' → * Unary Term'
      | / Unary Term'
      | λ

Unary → ! Unary
      | - Unary
      | Factor

Factor → ( Expr )
       | id G
       | num
       | str
       | true
       | false

G → ( H
  | λ

H → ArgList )
  | )

ArgList → Expr ArgList'

ArgList' → , Expr ArgList'
         | λ
```

## 2.1 Conjuntos Primeros

| No terminal | Primeros |
|---|---|
| `Program` | `{ int, float, bool, void, string }` |
| `DeclList` | `{ int, float, bool, void, string }` |
| `DeclList'` | `{ int, float, bool, void, string, λ }` |
| `Decl` | `{ int, float, bool, void, string }` |
| `C` | `{ (, ;, = }` |
| `A` | `{ ;, = }` |
| `VarDecl` | `{ int, float, bool, void, string }` |
| `B` | `{ int, float, bool, void, string, ) }` |
| `Type` | `{ int, float, bool, void, string }` |
| `ParamList` | `{ int, float, bool, void, string }` |
| `ParamList'` | `{ ,, λ }` |
| `Param` | `{ int, float, bool, void, string }` |
| `Block` | `{ { }` |
| `StmtList` | `{ int, float, bool, void, string, id, if, while, for, return, print, read, λ }` |
| `StmtList'` | `{ int, float, bool, void, string, id, if, while, for, return, print, read, λ }` |
| `Stmt` | `{ int, float, bool, void, string, id, if, while, for, return, print, read }` |
| `I` | `{ =, ( }` |
| `K` | `{ !, -, (, id, num, str, true, false, ) }` |
| `J` | `{ !, -, (, id, num, str, true, false, ; }` |
| `IfStmt` | `{ if }` |
| `D` | `{ else, λ }` |
| `WhileStmt` | `{ while }` |
| `ForStmt` | `{ for }` |
| `Expr` | `{ !, -, (, id, num, str, true, false }` |
| `Expr'` | `{ \|\|, &&, λ }` |
| `RelExpr` | `{ !, -, (, id, num, str, true, false }` |
| `E` | `{ ==, !=, <, >, λ }` |
| `RelOp` | `{ ==, !=, <, > }` |
| `F` | `{ =, λ }` |
| `AddExpr` | `{ !, -, (, id, num, str, true, false }` |
| `AddExpr'` | `{ +, -, λ }` |
| `Term` | `{ !, -, (, id, num, str, true, false }` |
| `Term'` | `{ *, /, λ }` |
| `Unary` | `{ !, -, (, id, num, str, true, false }` |
| `Factor` | `{ (, id, num, str, true, false }` |
| `G` | `{ (, λ }` |
| `H` | `{ !, -, (, id, num, str, true, false, ) }` |
| `ArgList` | `{ !, -, (, id, num, str, true, false }` |
| `ArgList'` | `{ ,, λ }` |

3. Siguientes
Para calcular los "Siguientes" usamos las siguientes reglas:

Regla 1
Si A es el símbolo inicial, entonces:
# ∈ FOLLOW(A)

Regla 2
Si existe una producción:
B → α A β
entonces:
Primeros(β) - {λ} ⊆ Siguientes(A)

Regla 3
Si existe una producción:
B → α A
o:
B → α A β
donde:
β ⇒* λ
entonces:
Siguientes(B) ⊆ Siguientes(A)



## 2.3 Conjuntos Siguientes

| No terminal | Siguientes |
|---|---|
| `Program` | `{ # }` |
| `DeclList` | `{ # }` |
| `DeclList'` | `{ # }` |
| `Decl` | `{ int, float, bool, void, string, # }` |
| `Type` | `{ id }` |
| `C` | `{ int, float, bool, void, string, # }` |
| `A` | `{ int, float, bool, void, string, id, if, while, for, return, print, read, }, # }` |
| `B` | `{ int, float, bool, void, string, # }` |
| `VarDecl` | `{ int, float, bool, void, string, id, if, while, for, return, print, read, } }` |
| `ParamList` | `{ ) }` |
| `ParamList'` | `{ ) }` |
| `Param` | `{ ,, ) }` |
| `Block` | `{ else, int, float, bool, void, string, id, if, while, for, return, print, read, }, # }` |
| `StmtList` | `{ } }` |
| `StmtList'` | `{ } }` |
| `Stmt` | `{ int, float, bool, void, string, id, if, while, for, return, print, read, } }` |
| `I` | `{ int, float, bool, void, string, id, if, while, for, return, print, read, } }` |
| `K` | `{ int, float, bool, void, string, id, if, while, for, return, print, read, } }` |
| `J` | `{ int, float, bool, void, string, id, if, while, for, return, print, read, } }` |
| `IfStmt` | `{ int, float, bool, void, string, id, if, while, for, return, print, read, } }` |
| `D` | `{ int, float, bool, void, string, id, if, while, for, return, print, read, } }` |
| `WhileStmt` | `{ int, float, bool, void, string, id, if, while, for, return, print, read, } }` |
| `ForStmt` | `{ int, float, bool, void, string, id, if, while, for, return, print, read, } }` |
| `Expr` | `{ ), ,, ; }` |
| `Expr'` | `{ ), ,, ; }` |
| `RelExpr` | `{ ||, &&, ), ,, ; }` |
| `E` | `{ ||, &&, ), ,, ; }` |
| `RelOp` | `{ !, -, (, id, num, str, true, false }` |
| `F` | `{ !, -, (, id, num, str, true, false }` |
| `AddExpr` | `{ ==, !=, <, >, ||, &&, ), ,, ; }` |
| `AddExpr'` | `{ ==, !=, <, >, ||, &&, ), ,, ; }` |
| `Term` | `{ +, -, ==, !=, <, >, ||, &&, ), ,, ; }` |
| `Term'` | `{ +, -, ==, !=, <, >, ||, &&, ), ,, ; }` |
| `Unary` | `{ *, /, +, -, ==, !=, <, >, ||, &&, ), ,, ; }` |
| `Factor` | `{ *, /, +, -, ==, !=, <, >, ||, &&, ), ,, ; }` |
| `G` | `{ *, /, +, -, ==, !=, <, >, ||, &&, ), ,, ; }` |
| `H` | `{ *, /, +, -, ==, !=, <, >, ||, &&, ), ,, ; }` |
| `ArgList` | `{ ) }` |
| `ArgList'` | `{ ) }` |







## 3. Símbolos directrices

Los símbolos directrices permiten determinar qué producción debe seleccionar el analizador sintáctico predictivo cuando se encuentra un determinado no terminal y observa el próximo token de entrada.

Para una producción A → α se utilizan las siguientes reglas:

- Si λ no pertenece a FIRST(α), entonces:

  DS(A → α) = FIRST(α)

- Si λ pertenece a FIRST(α), entonces:

  DS(A → α) = (FIRST(α) - {λ}) ∪ FOLLOW(A)

En particular, para una producción A → λ:

  DS(A → λ) = FOLLOW(A)

Estos conjuntos se utilizarán posteriormente para construir la tabla de análisis sintáctico predictivo. Para que la gramática sea LL(1), las producciones alternativas de un mismo no terminal deben tener conjuntos de símbolos directrices disjuntos.

### 3.1. Cálculo de los símbolos directrices

Los símbolos directrices de las producciones de la gramática se calculan a partir de los conjuntos FIRST y FOLLOW obtenidos anteriormente.

| Producción | Símbolos directrices |
|---|---|
| Program → DeclList | { int, float, bool, void, string } |
| DeclList → Decl DeclList' | { int, float, bool, void, string } |
| DeclList' → Decl DeclList' | { int, float, bool, void, string } |
| DeclList' → λ | { # } |
| Decl → Type id C | { int, float, bool, void, string } |
| C → ( B | { ( } |
| C → A | { ;, = } |
| A → ; | { ; } |
| A → = Expr ; | { = } |
| Type → int | { int } |
| Type → float | { float } |
| Type → bool | { bool } |
| Type → void | { void } |
| Type → string | { string } |
| B → ParamList ) Block | { int, float, bool, void, string } |
| B → ) Block | { ) } |
| ParamList → Param ParamList' | { int, float, bool, void, string } |
| ParamList' → , Param ParamList' | { , } |
| ParamList' → λ | { ) } |
| Param → Type id | { int, float, bool, void, string } |
| Block → { StmtList } | { { } |
| StmtList → Stmt StmtList' | { int, float, bool, void, string, id, if, while, for, return, print, read } |
| StmtList → λ | { } } |
| StmtList' → Stmt StmtList' | { int, float, bool, void, string, id, if, while, for, return, print, read } |
| StmtList' → λ | { } } |
| Stmt → VarDecl | { int, float, bool, void, string } |
| Stmt → id I | { id } |
| Stmt → IfStmt | { if } |
| Stmt → WhileStmt | { while } |
| Stmt → ForStmt | { for } |
| Stmt → return J | { return } |
| Stmt → print ( Expr ) ; | { print } |
| Stmt → read ( id ) ; | { read } |
| I → = Expr ; | { = } |
| I → ( K | { ( } |
| K → ArgList ) ; | { !, -, (, id, num, str, true, false } |
| K → ) ; | { ) } |
| J → Expr ; | { !, -, (, id, num, str, true, false } |
| J → ; | { ; } |
| IfStmt → if ( Expr ) Block D | { if } |
| D → else Block | { else } |
| D → λ | { int, float, bool, void, string, id, if, while, for, return, print, read, } } |
| WhileStmt → while ( Expr ) Block | { while } |
| ForStmt → for ( id = Expr ; Expr ; id = Expr ) Block | { for } |
| Expr → RelExpr Expr' | { !, -, (, id, num, str, true, false } |
| Expr' → \|\| RelExpr Expr' | { \|\| } |
| Expr' → && RelExpr Expr' | { && } |
| Expr' → λ | { ), ,, ; } |
| RelExpr → AddExpr E | { !, -, (, id, num, str, true, false } |
| E → RelOp AddExpr | { ==, !=, <, > } |
| E → λ | { \|\|, &&, ), ,, ; } |
| RelOp → == | { == } |
| RelOp → != | { != } |
| RelOp → < F | { < } |
| RelOp → > F | { > } |
| F → = | { = } |
| F → λ | { !, -, (, id, num, str, true, false } |
| AddExpr → Term AddExpr' | { !, -, (, id, num, str, true, false } |
| AddExpr' → + Term AddExpr' | { + } |
| AddExpr' → - Term AddExpr' | { - } |
| AddExpr' → λ | { ==, !=, <, >, \|\|, &&, ), ,, ; } |
| Term → Unary Term' | { !, -, (, id, num, str, true, false } |
| Term' → * Unary Term' | { * } |
| Term' → / Unary Term' | { / } |
| Term' → λ | { +, -, ==, !=, <, >, \|\|, &&, ), ,, ; } |
| Unary → ! Unary | { ! } |
| Unary → - Unary | { - } |
| Unary → Factor | { (, id, num, str, true, false } |
| Factor → ( Expr ) | { ( } |
| Factor → id G | { id } |
| Factor → num | { num } |
| Factor → str | { str } |
| Factor → true | { true } |
| Factor → false | { false } |
| G → ( H | { ( } |
| G → λ | { *, /, +, -, ==, !=, <, >, \|\|, &&, ), ,, ; } |
| H → ArgList ) | { !, -, (, id, num, str, true, false } |
| H → ) | { ) } |
| ArgList → Expr ArgList' | { !, -, (, id, num, str, true, false } |
| ArgList' → , Expr ArgList' | { , } |
| ArgList' → λ | { ) } |

### 3.2. Verificación de conflictos LL(1)

Para que una gramática sea LL(1), las producciones alternativas de un mismo no terminal deben tener conjuntos de símbolos directrices disjuntos.

Al comparar los símbolos directrices de las producciones alternativas de cada no terminal, no se presentan intersecciones entre las alternativas.

Por lo tanto, la gramática modificada no presenta conflictos en los símbolos directrices y puede ser utilizada para construir una tabla de análisis sintáctico predictivo LL(1).



## 4. Tabla de análisis sintáctico predictivo

La tabla de análisis sintáctico predictivo permite determinar qué producción debe aplicar el analizador sintáctico a partir del no terminal que se encuentra en la cima de la pila y del próximo token de entrada.

La tabla se representa como M[A, a], donde:

- A es un no terminal de la gramática.
- a es un terminal o el símbolo de fin de entrada `#`.
- M[A, a] contiene la producción que debe utilizarse.

Para construir la tabla se utilizan los símbolos directrices calculados anteriormente. Para cada producción A → α, se coloca dicha producción en todas las columnas correspondientes a los símbolos pertenecientes a su conjunto de símbolos directrices.

Si una celda de la tabla queda vacía, significa que no existe una producción válida para esa combinación de no terminal y token de entrada.

En una gramática LL(1), cada celda de la tabla debe contener como máximo una producción. Si una misma celda recibe dos producciones diferentes, existe un conflicto y la gramática no es LL(1).

### 4.1. Construcción de la tabla

La tabla predictiva se construye utilizando los conjuntos de símbolos directrices obtenidos en la sección anterior.

Por ejemplo:

- Para `Program → DeclList`, los símbolos directrices son `{ int, float, bool, void, string }`. Por lo tanto, esa producción se coloca en las columnas correspondientes a esos cinco tokens.
- Para `DeclList' → λ`, el conjunto de símbolos directrices es `{ # }`, por lo que la producción se coloca en la columna `#`.
- Para `Expr' → λ`, los símbolos directrices son `{ ), ,, ; }`, por lo que la producción se coloca en cada una de esas columnas.

De esta manera se completa la tabla para todos los no terminales de la gramática.

### 4.2. Tabla predictiva

A partir de los símbolos directrices obtenidos anteriormente, se construyen las entradas de la tabla M[A, a].

En la siguiente tabla se muestran las celdas que contienen una producción. Los tokens que no aparecen para un determinado no terminal representan celdas de error.

| No terminal | Token de entrada | Producción |
|---|---|---|
| Program | int, float, bool, void, string | Program → DeclList |
| DeclList | int, float, bool, void, string | DeclList → Decl DeclList' |
| DeclList' | int, float, bool, void, string | DeclList' → Decl DeclList' |
| DeclList' | # | DeclList' → λ |
| Decl | int, float, bool, void, string | Decl → Type id C |
| C | ( | C → ( B |
| C | ;, = | C → A |
| A | ; | A → ; |
| A | = | A → = Expr ; |
| Type | int | Type → int |
| Type | float | Type → float |
| Type | bool | Type → bool |
| Type | void | Type → void |
| Type | string | Type → string |
| B | int, float, bool, void, string | B → ParamList ) Block |
| B | ) | B → ) Block |
| ParamList | int, float, bool, void, string | ParamList → Param ParamList' |
| ParamList' | , | ParamList' → , Param ParamList' |
| ParamList' | ) | ParamList' → λ |
| Param | int, float, bool, void, string | Param → Type id |
| Block | { | Block → { StmtList } |
| StmtList | int, float, bool, void, string, id, if, while, for, return, print, read | StmtList → Stmt StmtList' |
| StmtList | } | StmtList → λ |
| StmtList' | int, float, bool, void, string, id, if, while, for, return, print, read | StmtList' → Stmt StmtList' |
| StmtList' | } | StmtList' → λ |
| Stmt | int, float, bool, void, string | Stmt → VarDecl |
| Stmt | id | Stmt → id I |
| Stmt | if | Stmt → IfStmt |
| Stmt | while | Stmt → WhileStmt |
| Stmt | for | Stmt → ForStmt |
| Stmt | return | Stmt → return J |
| Stmt | print | Stmt → print ( Expr ) ; |
| Stmt | read | Stmt → read ( id ) ; |
| I | = | I → = Expr ; |
| I | ( | I → ( K |
| K | !, -, (, id, num, str, true, false | K → ArgList ) ; |
| K | ) | K → ) ; |
| J | !, -, (, id, num, str, true, false | J → Expr ; |
| J | ; | J → ; |
| IfStmt | if | IfStmt → if ( Expr ) Block D |
| D | else | D → else Block |
| D | int, float, bool, void, string, id, if, while, for, return, print, read, } | D → λ |
| WhileStmt | while | WhileStmt → while ( Expr ) Block |
| ForStmt | for | ForStmt → for ( id = Expr ; Expr ; id = Expr ) Block |
| Expr | !, -, (, id, num, str, true, false | Expr → RelExpr Expr' |
| Expr' | \|\| | Expr' → \|\| RelExpr Expr' |
| Expr' | && | Expr' → && RelExpr Expr' |
| Expr' | ), ,, ; | Expr' → λ |
| RelExpr | !, -, (, id, num, str, true, false | RelExpr → AddExpr E |
| E | ==, !=, <, > | E → RelOp AddExpr |
| E | \|\|, &&, ), ,, ; | E → λ |
| RelOp | == | RelOp → == |
| RelOp | != | RelOp → != |
| RelOp | < | RelOp → < F |
| RelOp | > | RelOp → > F |
| F | = | F → = |
| F | !, -, (, id, num, str, true, false | F → λ |
| AddExpr | !, -, (, id, num, str, true, false | AddExpr → Term AddExpr' |
| AddExpr' | + | AddExpr' → + Term AddExpr' |
| AddExpr' | - | AddExpr' → - Term AddExpr' |
| AddExpr' | ==, !=, <, >, \|\|, &&, ), ,, ; | AddExpr' → λ |
| Term | !, -, (, id, num, str, true, false | Term → Unary Term' |
| Term' | * | Term' → * Unary Term' |
| Term' | / | Term' → / Unary Term' |
| Term' | +, -, ==, !=, <, >, \|\|, &&, ), ,, ; | Term' → λ |
| Unary | ! | Unary → ! Unary |
| Unary | - | Unary → - Unary |
| Unary | (, id, num, str, true, false | Unary → Factor |
| Factor | ( | Factor → ( Expr ) |
| Factor | id | Factor → id G |
| Factor | num | Factor → num |
| Factor | str | Factor → str |
| Factor | true | Factor → true |
| Factor | false | Factor → false |
| G | ( | G → ( H |
| G | *, /, +, -, ==, !=, <, >, \|\|, &&, ), ,, ; | G → λ |
| H | !, -, (, id, num, str, true, false | H → ArgList ) |
| H | ) | H → ) |
| ArgList | !, -, (, id, num, str, true, false | ArgList → Expr ArgList' |
| ArgList' | , | ArgList' → , Expr ArgList' |
| ArgList' | ) | ArgList' → λ |

Las celdas no especificadas en la tabla representan situaciones de error sintáctico, ya que no existe una producción aplicable para esa combinación de no terminal y token de entrada.





## 5. Algoritmo de análisis sintáctico con tabla

El analizador sintáctico descendente predictivo por tabla utiliza una pila y la tabla de análisis sintáctico para determinar qué producción aplicar.

La cadena de entrada se completa agregando el símbolo de fin de entrada #. La pila se inicializa con dos símbolos: # en el fondo y el símbolo distinguido de la gramática, Program, en el tope.

En cada paso se considera el símbolo t apuntado en la cadena de entrada, el tope de la pila y la tabla M.

El algoritmo es el siguiente:

1. Si el tope de la pila es un terminal:
   - Si el tope coincide con t, se realiza Pop() y se avanza al siguiente símbolo de entrada.
   - Si no coincide, se informa un error.

2. Si el tope de la pila es un no terminal:
   - Se consulta la tabla M utilizando el símbolo de entrada y el no terminal del tope.
   - Si la tabla contiene una producción A → Y1 Y2 ... Yk, se realiza Pop() y luego se agregan a la pila los símbolos Yk, ..., Y2, Y1, de manera que Y1 quede en el tope.
   - Si no existe una producción correspondiente, se informa un error.

3. El análisis continúa hasta que tanto el tope de la pila como el símbolo de entrada sean #.

Este procedimiento permite realizar el análisis de forma descendente y predictiva, seleccionando una única producción a partir del no terminal en el tope de la pila y del siguiente símbolo de entrada.

### 5.1. Pseudocódigo del algoritmo

```text
t ← primer símbolo de la entrada
Pila ← [#, Program]

repeat
    if tope ∈ VT then
        if tope == t then
            Pop()
            t ← t + 1
        else
            return Error
        end if
    else
        if M(tope, t) == tope → Y1 Y2 ... Yk then
            Pop()
            Push(Yk, ..., Y2, Y1)
        else
            return Error
        end if
    end if
until tope == # ∧ t == #

In [21]:
#lexer TP1
lista_afds = [ 
    ("INT",
    0, 
    { 
        (0, 'i'): 1, 
        (1, 'n'): 2, 
        (2, 't'): 3 
    }, 
    {3} 
    ), 
    ("FLOAT", 
    0, 
    { 
        (0, 'f'): 1, 
        (1, 'l'): 2, 
        (2, 'o'): 3, 
        (3, 'a'): 4, 
        (4, 't'): 5 
    }, 
    {5} 
    ), 
    ("BOOL", 
    0, 
    { 
        (0, 'b'): 1, 
        (1, 'o'): 2, 
        (2, 'o'): 3, 
        (3, 'l'): 4 
    }, 
    {4} 
    ), 
    ("VOID", 
    0, 
    { 
        (0, 'v'): 1, 
        (1, 'o'): 2, 
        (2, 'i'): 3, 
        (3, 'd'): 4 
    }, 
    {4} 
    ), 
    ("IF", 
    0, 
    { 
        (0, 'i'): 1, 
        (1, 'f'): 2 
    }, 
    {2} 
    ), 
    ("ELSE", 
    0, 
    { 
        (0, 'e'): 1, 
        (1, 'l'): 2, 
        (2, 's'): 3, 
        (3, 'e'): 4 
    }, 
    {4} 
    ), 
    ("WHILE", 
    0, 
    { 
        (0, 'w'): 1, 
        (1, 'h'): 2, 
        (2, 'i'): 3, 
        (3, 'l'): 4, 
        (4, 'e'): 5 
    }, 
    {5} 
    ), 
    ("FOR", 
    0, 
    { 
        (0, 'f'): 1, 
        (1, 'o'): 2, 
        (2, 'r'): 3 
    }, 
    {3} 
    ), 
    ("READ", 
    0, 
    { 
        (0, 'r'): 1, 
        (1, 'e'): 2, 
        (2, 'a'): 3, 
        (3, 'd'): 4 
    }, 
    {4} 
    ), 
    ("RETURN", 
    0, 
    { 
        (0, 'r'): 1, 
        (1, 'e'): 2, 
        (2, 't'): 3, 
        (3, 'u'): 4, 
        (4, 'r'): 5, 
        (5, 'n'): 6 
    }, 
    {6} 
    ), 
    ("PRINT", 
    0, 
    { 
        (0, 'p'): 1, 
        (1, 'r'): 2, 
        (2, 'i'): 3, 
        (3, 'n'): 4, 
        (4, 't'): 5 
    }, 
    {5} 
    ), 
    ("TRUE", 
    0, 
    { 
        (0, 't'): 1, 
        (1, 'r'): 2, 
        (2, 'u'): 3, 
        (3, 'e'): 4 
    }, 
    {4} 
    ), 
    ("FALSE", 
    0, 
    { 
        (0, 'f'): 1, 
        (1, 'a'): 2, 
        (2, 'l'): 3, 
        (3, 's'): 4, 
        (4, 'e'): 5 
    }, 
    {5} 
    ), 
    ("IGUAL", 
    0, 
    { 
        (0, '='): 1, 
        (1, '='): 2 
    }, 
    {2} 
    ), 
    ("DISTINTO", 
    0, 
    { 
        (0, '!'): 1, 
        (1, '='): 2 
    }, 
    {2} 
    ), 
    ("MENOR_IGUAL", 
    0, 
    { 
        (0, '<'): 1, 
        (1, '='): 2 
    }, 
    {2} 
    ), 
    ("MAYOR_IGUAL", 
    0, 
    { 
        (0, '>'): 1, 
        (1, '='): 2 
    }, 
    {2} 
    ), 
    ("AND", 
    0, 
    { 
        (0, '&'): 1, 
        (1, '&'): 2 
    }, 
    {2} 
    ), 
    ("OR", 
    0, 
    { 
        (0, '|'): 1, 
        (1, '|'): 2 
    }, 
    {2} 
    ), 
    ("ASIGNACION", 
    0, 
    { 
        (0, '='): 1 
    }, 
    {1} 
    ), 
    ("NOT", 
    0, 
    { 
        (0, '!'): 1 
    }, 
    {1} 
    ), 
    ("MENOR", 
    0, 
    { 
        (0, '<'): 1 
    }, 
    {1} 
    ), 
    ("MAYOR", 
    0, 
    { 
        (0, '>'): 1 
    }, 
    {1} 
    ), 
    ("SUMA", 
    0, 
    { 
        (0, '+'): 1 
    }, 
    {1} 
    ), 
    ("RESTA", 
    0, 
    { 
        (0, '-'): 1 
    }, 
    {1} 
    ), 
    ("MULT", 
    0, 
    { 
        (0, '*'): 1 
    }, 
    {1} 
    ), 
    ("DIV", 
    0, 
    { 
        (0, '/'): 1 
    }, 
    {1} 
    ),
    ("WH",
     0,
     {
         (0, ' '):1,
         (0, '\t'):1,
         (0, '\n'):1
     },
     {1}
    ), 
    ("PARENT_LEFT", 
    0, 
    { 
        (0, '('):1 
    }, 
    {1} 
    ), 
    ("PARENT_RIGHT", 
    0, 
    { 
        (0, ')'): 1 
    }, 
    {1} 
    ), 
    ("LLAVE_LEFT", 
    0, 
    { 
        (0, '{'): 1 
    }, 
    {1} 
    ), 
    ("LLAVE_RIGHT", 
    0, 
    { 
        (0, '}'): 1 
    }, 
    {1} 
    ), 
    ("COMMADOT", 
    0, 
    { 
        (0, ';'): 1 
    }, 
    {1} 
    ), 
    ("COMMA", 
    0, 
    { 
        (0, ','): 1 
    }, 
    {1} 
    ), 
    ("STRING", 
    0,
    {
        (0, '"'): 1,
        #Minusculas
        (1, 'a'): 1,
        (1, 'b'): 1,
        (1, 'c'): 1,
        (1, 'd'): 1,
        (1, 'e'): 1,
        (1, 'f'): 1,
        (1, 'g'): 1,
        (1, 'h'): 1,
        (1, 'i'): 1,
        (1, 'j'): 1,
        (1, 'k'): 1,
        (1, 'l'): 1,
        (1, 'm'): 1,
        (1, 'n'): 1,
        (1, 'ñ'): 1,
        (1, 'o'): 1,
        (1, 'p'): 1,
        (1, 'q'): 1,
        (1, 'r'): 1,
        (1, 's'): 1,
        (1, 't'): 1,
        (1, 'u'): 1,
        (1, 'v'): 1,
        (1, 'w'): 1,
        (1, 'x'): 1,
        (1, 'y'): 1,
        (1, 'z'): 1,
        #Mayusculas
        (1, 'A'): 1,
        (1, 'B'): 1,
        (1, 'C'): 1,
        (1, 'D'): 1,
        (1, 'E'): 1,
        (1, 'F'): 1,
        (1, 'G'): 1,
        (1, 'H'): 1,
        (1, 'I'): 1,
        (1, 'J'): 1,
        (1, 'K'): 1,
        (1, 'L'): 1,
        (1, 'M'): 1,
        (1, 'N'): 1,
        (1, 'Ñ'): 1,
        (1, 'O'): 1,
        (1, 'P'): 1,
        (1, 'Q'): 1,
        (1, 'R'): 1,
        (1, 'S'): 1,
        (1, 'T'): 1,
        (1, 'U'): 1,
        (1, 'V'): 1,
        (1, 'W'): 1,
        (1, 'X'): 1,
        (1, 'Y'): 1,
        (1, 'Z'): 1,
        #Numeros
        (1, '0'): 1,
        (1, '1'): 1,
        (1, '2'): 1,
        (1, '3'): 1,
        (1, '4'): 1,
        (1, '5'): 1,
        (1, '6'): 1,
        (1, '7'): 1,
        (1, '8'): 1,
        (1, '9'): 1,
        #Caracteres
        (1, ' '): 1,
        (1, '.'): 1,
        (1, '_'): 1,
        (1, '-'): 1,
        (1, '"'): 2
    },
    {2}
    ),  
    ("ID", 
    0, 
    {
        (0, 'a'): 1,
        (0, 'b'): 1,
        (0, 'c'): 1,
        (0, 'd'): 1,
        (0, 'e'): 1,
        (0, 'f'): 1,
        (0, 'g'): 1,
        (0, 'h'): 1,
        (0, 'i'): 1,
        (0, 'j'): 1,
        (0, 'k'): 1,
        (0, 'l'): 1,
        (0, 'm'): 1,
        (0, 'n'): 1,
        (0, 'ñ'): 1,
        (0, 'o'): 1,
        (0, 'p'): 1,
        (0, 'q'): 1,
        (0, 'r'): 1,
        (0, 's'): 1,
        (0, 't'): 1,
        (0, 'u'): 1,
        (0, 'v'): 1,
        (0, 'w'): 1,
        (0, 'x'): 1,
        (0, 'y'): 1,
        (0, 'z'): 1,
        #Mayusculas
        (0, 'A'): 1,
        (0, 'B'): 1,
        (0, 'C'): 1,
        (0, 'D'): 1,
        (0, 'E'): 1,
        (0, 'F'): 1,
        (0, 'G'): 1,
        (0, 'H'): 1,
        (0, 'I'): 1,
        (0, 'J'): 1,
        (0, 'K'): 1,
        (0, 'L'): 1,
        (0, 'M'): 1,
        (0, 'N'): 1,
        (0, 'Ñ'): 1,
        (0, 'O'): 1,
        (0, 'P'): 1,
        (0, 'Q'): 1,
        (0, 'R'): 1,
        (0, 'S'): 1,
        (0, 'T'): 1,
        (0, 'U'): 1,
        (0, 'V'): 1,
        (0, 'W'): 1,
        (0, 'X'): 1,
        (0, 'Y'): 1,
        (0, 'Z'): 1, 
        #
        (1, 'a'): 1,
        (1, 'b'): 1,
        (1, 'c'): 1,
        (1, 'd'): 1,
        (1, 'e'): 1,
        (1, 'f'): 1,
        (1, 'g'): 1,
        (1, 'h'): 1,
        (1, 'i'): 1,
        (1, 'j'): 1,
        (1, 'k'): 1,
        (1, 'l'): 1,
        (1, 'm'): 1,
        (1, 'n'): 1,
        (1, 'ñ'): 1,
        (1, 'o'): 1,
        (1, 'p'): 1,
        (1, 'q'): 1,
        (1, 'r'): 1,
        (1, 's'): 1,
        (1, 't'): 1,
        (1, 'u'): 1,
        (1, 'v'): 1,
        (1, 'w'): 1,
        (1, 'x'): 1,
        (1, 'y'): 1,
        (1, 'z'): 1,
        #Mayusculas
        (1, 'A'): 1,
        (1, 'B'): 1,
        (1, 'C'): 1,
        (1, 'D'): 1,
        (1, 'E'): 1,
        (1, 'F'): 1,
        (1, 'G'): 1,
        (1, 'H'): 1,
        (1, 'I'): 1,
        (1, 'J'): 1,
        (1, 'K'): 1,
        (1, 'L'): 1,
        (1, 'M'): 1,
        (1, 'N'): 1,
        (1, 'Ñ'): 1,
        (1, 'O'): 1,
        (1, 'P'): 1,
        (1, 'Q'): 1,
        (1, 'R'): 1,
        (1, 'S'): 1,
        (1, 'T'): 1,
        (1, 'U'): 1,
        (1, 'V'): 1,
        (1, 'W'): 1,
        (1, 'X'): 1,
        (1, 'Y'): 1,
        (1, 'Z'): 1,
        (1, '0'): 1,
        (1, '1'): 1,
        (1, '2'): 1,
        (1, '3'): 1,
        (1, '4'): 1,
        (1, '5'): 1,
        (1, '6'): 1,
        (1, '7'): 1,
        (1, '8'): 1,
        (1, '9'): 1,
        (1, '_'): 1,
    }, 
    {1}
    ), 
    ("NUMERO", 
    0, 
    {
        (0, '0'): 1,
        (0, '1'): 1, 
        (0, '2'): 1,
        (0, '3'): 1,
        (0, '4'): 1,
        (0, '5'): 1,
        (0, '6'): 1,
        (0, '7'): 1,
        (0, '8'): 1,
        (0, '9'): 1,
        (1, '0'): 1,
        (1, '1'): 1,
        (1, '2'): 1,
        (1, '3'): 1,
        (1, '4'): 1,
        (1, '5'): 1,
        (1, '6'): 1,
        (1, '7'): 1,
        (1, '8'): 1,
        (1, '9'): 1,
        (1, '.'): 2,
        (2, '0'): 2,
        (2, '1'): 2,
        (2, '2'): 2,
        (2, '3'): 2,
        (2, '4'): 2,
        (2, '5'): 2,
        (2, '6'): 2,
        (2, '7'): 2,
        (2, '8'): 2,
        (2, '9'): 2,
        
    }, 
    {1,2}
    ), 
    ]

def lexer_multiples_afds(codigo_fuente):
    tokens = []
    pos_actual = 0
    n = len(codigo_fuente)

    while pos_actual < n:
        longitud_mejor_match = 0
        tipo_mejor_match = None
        lexema_mejor_match = ''

        for afd in lista_afds:
            tipo, estado_inicial, delta, estados_aceptados = afd
            estado_actual = estado_inicial
            pos_lexema_actual = pos_actual
            ultima_pos_aceptada = -1

            while pos_lexema_actual < n:
                clave = (estado_actual, codigo_fuente[pos_lexema_actual])
                if clave not in delta:
                    break
                estado_actual = delta[clave]
                pos_lexema_actual += 1
                if estado_actual in estados_aceptados:
                    ultima_pos_aceptada = pos_lexema_actual

            if ultima_pos_aceptada > pos_actual:
                longitud_lexema_actual = ultima_pos_aceptada - pos_actual
                if longitud_lexema_actual > longitud_mejor_match:
                    longitud_mejor_match = longitud_lexema_actual
                    tipo_mejor_match = tipo
                    lexema_mejor_match = codigo_fuente[pos_actual:ultima_pos_aceptada]

        if longitud_mejor_match == 0:
            caracter = repr(codigo_fuente[pos_actual])
            raise ValueError(f"Carácter inesperado {caracter} en la posición {pos_actual}")
        if tipo_mejor_match != "WH":
            tokens.append((tipo_mejor_match, lexema_mejor_match))
            
        pos_actual += longitud_mejor_match

    tokens.append(("EOF", "EOF"))
    return tokens


In [ ]:
#Parser
#from main import lexer_multiples_afds linea para usar fuera del ipynb

VN = ['Program', 'DeclList', 'DeclList1', 'Decl', 'C', 'A', 'VarDecl', 'FuncDecl', 'B', 'Type', 'ParamList', 'ParamList1', 'Param', 'Block',
      'StmtList', 'StmtList1', 'Stmt', 'I', 'K', 'J', 'IfStmt', 'D', 'WhileStmt', 'ForStmt', 'Expr', 'Expr1', 'RelExpr', 'E', 'RelOp', 'F',
      'AddExpr', 'AddExpr1', 'Term', 'Term1', 'Unary', 'Factor', 'G', 'H', 'ArgList', 'ArgList1']

VT = ['INT', 'FLOAT', 'BOOL', 'VOID', 'IF', 'ELSE', 'WHILE', 'FOR', 'RETURN', 'READ', 'PRINT', 'TRUE', 'FALSE', 'IGUAL', 'DISTINTO',
      'MENOR_IGUAL', 'MAYOR_IGUAL', 'AND', 'OR', 'ASIGNACION', 'NOT', 'MENOR', 'MAYOR', 'SUMA', 'RESTA', 'MULT', 'DIV', 'PARENT_LEFT',
      'PARENT_RIGHT', 'LLAVE_LEFT', 'LLAVE_RIGHT', 'COMMADOT', 'COMMA', 'STRING', 'ID', 'NUMERO']

P = {'Program': [['DeclList']],
     'DeclList': [['Decl', 'DeclList1']],
     'DeclList1': [['Decl', 'DeclList1'], []],
     'Decl': [['Type', 'ID', 'C']],
     'C': [['PARENT_LEFT', 'B'], ['A']],
     'A': [['COMMADOT'], ['ASIGNACION', 'Expr', 'COMMADOT']],
     'VarDecl': [['Type', 'ID', 'A']],
     'FuncDecl': [['Type', 'ID', 'PARENT_LEFT', 'B']],
     'B': [['ParamList', 'PARENT_RIGHT', 'Block'], ['PARENT_RIGHT', 'Block']],
     'Type': [['INT'], ['FLOAT'], ['BOOL'], ['VOID'], ['STRING']],
     'ParamList': [['Param', 'ParamList1']],
     'ParamList1': [['COMMA', 'Param', 'ParamList1'], []],
     'Param': [['Type', 'ID']],
     'Block': [['LLAVE_LEFT', 'StmtList', 'LLAVE_RIGHT']],
     'StmtList': [['Stmt', 'StmtList1'], []],
     'StmtList1': [['Stmt', 'StmtList1'], []],
     'Stmt': [['VarDecl'], ['ID', 'I'], ['IfStmt'], ['WhileStmt'], ['ForStmt'], ['RETURN', 'J'],
              ['PRINT', 'PARENT_LEFT', 'Expr', 'PARENT_RIGHT', 'COMMADOT'],
              ['READ', 'PARENT_LEFT', 'ID', 'PARENT_RIGHT', 'COMMADOT']],
     'I': [['ASIGNACION', 'Expr', 'COMMADOT'], ['PARENT_LEFT', 'K']],
     'K': [['ArgList', 'PARENT_RIGHT', 'COMMADOT'], ['PARENT_RIGHT', 'COMMADOT']],
     'J': [['Expr', 'COMMADOT'], ['COMMADOT']],
     'IfStmt': [['IF', 'PARENT_LEFT', 'Expr', 'PARENT_RIGHT', 'Block', 'D']],
     'D': [['ELSE', 'Block'], []],
     'WhileStmt': [['WHILE', 'PARENT_LEFT', 'Expr', 'PARENT_RIGHT', 'Block']],
     'ForStmt': [['FOR', 'PARENT_LEFT', 'ID', 'ASIGNACION', 'Expr', 'COMMADOT', 'Expr', 'COMMADOT', 'ID', 'ASIGNACION', 'Expr', 'PARENT_RIGHT', 'Block']],
     'Expr': [['RelExpr', 'Expr1']],
     'Expr1': [['OR', 'RelExpr', 'Expr1'], ['AND', 'RelExpr', 'Expr1'], []],
     'RelExpr': [['AddExpr', 'E']],
     'E': [['RelOp', 'AddExpr'], []],
     'RelOp': [['IGUAL'], ['DISTINTO'], ['MENOR', 'F'], ['MAYOR', 'F'], ['MENOR_IGUAL'], ['MAYOR_IGUAL']],
     'F': [['ASIGNACION'], []],
     'AddExpr': [['Term', 'AddExpr1']],
     'AddExpr1': [['SUMA', 'Term', 'AddExpr1'], ['RESTA', 'Term', 'AddExpr1'], []],
     'Term': [['Unary', 'Term1']],
     'Term1': [['MULT', 'Unary', 'Term1'], ['DIV', 'Unary', 'Term1'], []],
     'Unary': [['NOT', 'Unary'], ['RESTA', 'Unary'], ['Factor']],
     'Factor': [['PARENT_LEFT', 'Expr', 'PARENT_RIGHT'], ['ID', 'G'], ['NUMERO'], ['STRING'], ['TRUE'], ['FALSE']],
     'G': [['PARENT_LEFT', 'H'], []],
     'H': [['ArgList', 'PARENT_RIGHT'], ['PARENT_RIGHT']],
     'ArgList': [['Expr', 'ArgList1']],
     'ArgList1': [['COMMA', 'Expr', 'ArgList1'], []]}

# M[(no_terminal, lookahead)] = indice de la produccion en P.
M = {
    ('Program', 'INT'): 0, ('Program', 'FLOAT'): 0, ('Program', 'BOOL'): 0, ('Program', 'VOID'): 0, ('Program', 'STRING'): 0,
    ('DeclList', 'INT'): 0, ('DeclList', 'FLOAT'): 0, ('DeclList', 'BOOL'): 0, ('DeclList', 'VOID'): 0, ('DeclList', 'STRING'): 0,
    ('DeclList1', 'INT'): 0, ('DeclList1', 'FLOAT'): 0, ('DeclList1', 'BOOL'): 0, ('DeclList1', 'VOID'): 0, ('DeclList1', 'STRING'): 0, ('DeclList1', 'EOF'): 1,
    ('Decl', 'INT'): 0, ('Decl', 'FLOAT'): 0, ('Decl', 'BOOL'): 0, ('Decl', 'VOID'): 0, ('Decl', 'STRING'): 0,
    ('C', 'PARENT_LEFT'): 0, ('C', 'COMMADOT'): 1, ('C', 'ASIGNACION'): 1,
    ('A', 'COMMADOT'): 0, ('A', 'ASIGNACION'): 1,
    ('VarDecl', 'INT'): 0, ('VarDecl', 'FLOAT'): 0, ('VarDecl', 'BOOL'): 0, ('VarDecl', 'VOID'): 0, ('VarDecl', 'STRING'): 0,
    ('B', 'INT'): 0, ('B', 'FLOAT'): 0, ('B', 'BOOL'): 0, ('B', 'VOID'): 0, ('B', 'STRING'): 0, ('B', 'PARENT_RIGHT'): 1,
    ('Type', 'INT'): 0, ('Type', 'FLOAT'): 1, ('Type', 'BOOL'): 2, ('Type', 'VOID'): 3, ('Type', 'STRING'): 4,
    ('ParamList', 'INT'): 0, ('ParamList', 'FLOAT'): 0, ('ParamList', 'BOOL'): 0, ('ParamList', 'VOID'): 0, ('ParamList', 'STRING'): 0,
    ('ParamList1', 'COMMA'): 0, ('ParamList1', 'PARENT_RIGHT'): 1,
    ('Param', 'INT'): 0, ('Param', 'FLOAT'): 0, ('Param', 'BOOL'): 0, ('Param', 'VOID'): 0, ('Param', 'STRING'): 0,
    ('Block', 'LLAVE_LEFT'): 0,
    ('StmtList', 'INT'): 0, ('StmtList', 'FLOAT'): 0, ('StmtList', 'BOOL'): 0, ('StmtList', 'VOID'): 0, ('StmtList', 'STRING'): 0,
    ('StmtList', 'ID'): 0, ('StmtList', 'IF'): 0, ('StmtList', 'WHILE'): 0, ('StmtList', 'FOR'): 0, ('StmtList', 'RETURN'): 0,
    ('StmtList', 'PRINT'): 0, ('StmtList', 'READ'): 0, ('StmtList', 'LLAVE_RIGHT'): 1,
    ('StmtList1', 'INT'): 0, ('StmtList1', 'FLOAT'): 0, ('StmtList1', 'BOOL'): 0, ('StmtList1', 'VOID'): 0, ('StmtList1', 'STRING'): 0,
    ('StmtList1', 'ID'): 0, ('StmtList1', 'IF'): 0, ('StmtList1', 'WHILE'): 0, ('StmtList1', 'FOR'): 0, ('StmtList1', 'RETURN'): 0,
    ('StmtList1', 'PRINT'): 0, ('StmtList1', 'READ'): 0, ('StmtList1', 'LLAVE_RIGHT'): 1,
    ('Stmt', 'INT'): 0, ('Stmt', 'FLOAT'): 0, ('Stmt', 'BOOL'): 0, ('Stmt', 'VOID'): 0, ('Stmt', 'STRING'): 0,
    ('Stmt', 'ID'): 1, ('Stmt', 'IF'): 2, ('Stmt', 'WHILE'): 3, ('Stmt', 'FOR'): 4, ('Stmt', 'RETURN'): 5, ('Stmt', 'PRINT'): 6, ('Stmt', 'READ'): 7,
    ('I', 'ASIGNACION'): 0, ('I', 'PARENT_LEFT'): 1,
    ('K', 'NOT'): 0, ('K', 'RESTA'): 0, ('K', 'PARENT_LEFT'): 0, ('K', 'ID'): 0, ('K', 'NUMERO'): 0, ('K', 'STRING'): 0, ('K', 'TRUE'): 0, ('K', 'FALSE'): 0, ('K', 'PARENT_RIGHT'): 1,
    ('J', 'NOT'): 0, ('J', 'RESTA'): 0, ('J', 'PARENT_LEFT'): 0, ('J', 'ID'): 0, ('J', 'NUMERO'): 0, ('J', 'STRING'): 0, ('J', 'TRUE'): 0, ('J', 'FALSE'): 0, ('J', 'COMMADOT'): 1,
    ('IfStmt', 'IF'): 0,
    ('D', 'ELSE'): 0, ('D', 'INT'): 1, ('D', 'FLOAT'): 1, ('D', 'BOOL'): 1, ('D', 'VOID'): 1, ('D', 'STRING'): 1,
    ('D', 'ID'): 1, ('D', 'IF'): 1, ('D', 'WHILE'): 1, ('D', 'FOR'): 1, ('D', 'RETURN'): 1, ('D', 'PRINT'): 1, ('D', 'READ'): 1, ('D', 'LLAVE_RIGHT'): 1,
    ('WhileStmt', 'WHILE'): 0, ('ForStmt', 'FOR'): 0,
    ('Expr', 'NOT'): 0, ('Expr', 'RESTA'): 0, ('Expr', 'PARENT_LEFT'): 0, ('Expr', 'ID'): 0, ('Expr', 'NUMERO'): 0, ('Expr', 'STRING'): 0, ('Expr', 'TRUE'): 0, ('Expr', 'FALSE'): 0,
    ('Expr1', 'OR'): 0, ('Expr1', 'AND'): 1, ('Expr1', 'COMMADOT'): 2, ('Expr1', 'COMMA'): 2, ('Expr1', 'PARENT_RIGHT'): 2,
    ('RelExpr', 'NOT'): 0, ('RelExpr', 'RESTA'): 0, ('RelExpr', 'PARENT_LEFT'): 0, ('RelExpr', 'ID'): 0, ('RelExpr', 'NUMERO'): 0, ('RelExpr', 'STRING'): 0, ('RelExpr', 'TRUE'): 0, ('RelExpr', 'FALSE'): 0,
    ('E', 'IGUAL'): 0, ('E', 'DISTINTO'): 0, ('E', 'MENOR'): 0, ('E', 'MAYOR'): 0, ('E', 'MENOR_IGUAL'): 0, ('E', 'MAYOR_IGUAL'): 0,
    ('E', 'OR'): 1, ('E', 'AND'): 1, ('E', 'COMMADOT'): 1, ('E', 'COMMA'): 1, ('E', 'PARENT_RIGHT'): 1,
    ('RelOp', 'IGUAL'): 0, ('RelOp', 'DISTINTO'): 1, ('RelOp', 'MENOR'): 2, ('RelOp', 'MAYOR'): 3, ('RelOp', 'MENOR_IGUAL'): 4, ('RelOp', 'MAYOR_IGUAL'): 5,
    ('F', 'ASIGNACION'): 0, ('F', 'NOT'): 1, ('F', 'RESTA'): 1, ('F', 'PARENT_LEFT'): 1, ('F', 'ID'): 1, ('F', 'NUMERO'): 1, ('F', 'STRING'): 1, ('F', 'TRUE'): 1, ('F', 'FALSE'): 1,
    ('AddExpr', 'NOT'): 0, ('AddExpr', 'RESTA'): 0, ('AddExpr', 'PARENT_LEFT'): 0, ('AddExpr', 'ID'): 0, ('AddExpr', 'NUMERO'): 0, ('AddExpr', 'STRING'): 0, ('AddExpr', 'TRUE'): 0, ('AddExpr', 'FALSE'): 0,
    ('AddExpr1', 'SUMA'): 0, ('AddExpr1', 'RESTA'): 1, ('AddExpr1', 'IGUAL'): 2, ('AddExpr1', 'DISTINTO'): 2, ('AddExpr1', 'MENOR'): 2, ('AddExpr1', 'MAYOR'): 2,
    ('AddExpr1', 'MENOR_IGUAL'): 2, ('AddExpr1', 'MAYOR_IGUAL'): 2, ('AddExpr1', 'OR'): 2, ('AddExpr1', 'AND'): 2, ('AddExpr1', 'PARENT_RIGHT'): 2, ('AddExpr1', 'COMMA'): 2, ('AddExpr1', 'COMMADOT'): 2,
    ('Term', 'NOT'): 0, ('Term', 'RESTA'): 0, ('Term', 'PARENT_LEFT'): 0, ('Term', 'ID'): 0, ('Term', 'NUMERO'): 0, ('Term', 'STRING'): 0, ('Term', 'TRUE'): 0, ('Term', 'FALSE'): 0,
    ('Term1', 'MULT'): 0, ('Term1', 'DIV'): 1, ('Term1', 'SUMA'): 2, ('Term1', 'RESTA'): 2, ('Term1', 'IGUAL'): 2, ('Term1', 'DISTINTO'): 2, ('Term1', 'MENOR'): 2, ('Term1', 'MAYOR'): 2,
    ('Term1', 'MENOR_IGUAL'): 2, ('Term1', 'MAYOR_IGUAL'): 2, ('Term1', 'OR'): 2, ('Term1', 'AND'): 2, ('Term1', 'PARENT_RIGHT'): 2, ('Term1', 'COMMA'): 2, ('Term1', 'COMMADOT'): 2,
    ('Unary', 'NOT'): 0, ('Unary', 'RESTA'): 1, ('Unary', 'PARENT_LEFT'): 2, ('Unary', 'ID'): 2, ('Unary', 'NUMERO'): 2, ('Unary', 'STRING'): 2, ('Unary', 'TRUE'): 2, ('Unary', 'FALSE'): 2,
    ('Factor', 'PARENT_LEFT'): 0, ('Factor', 'ID'): 1, ('Factor', 'NUMERO'): 2, ('Factor', 'STRING'): 3, ('Factor', 'TRUE'): 4, ('Factor', 'FALSE'): 5,
    ('G', 'PARENT_LEFT'): 0, ('G', 'MULT'): 1, ('G', 'DIV'): 1, ('G', 'SUMA'): 1, ('G', 'RESTA'): 1, ('G', 'IGUAL'): 1, ('G', 'DISTINTO'): 1, ('G', 'MENOR'): 1,
    ('G', 'MAYOR'): 1, ('G', 'MENOR_IGUAL'): 1, ('G', 'MAYOR_IGUAL'): 1, ('G', 'OR'): 1, ('G', 'AND'): 1, ('G', 'PARENT_RIGHT'): 1, ('G', 'COMMA'): 1, ('G', 'COMMADOT'): 1,
    ('H', 'NOT'): 0, ('H', 'RESTA'): 0, ('H', 'PARENT_LEFT'): 0, ('H', 'ID'): 0, ('H', 'NUMERO'): 0, ('H', 'STRING'): 0, ('H', 'TRUE'): 0, ('H', 'FALSE'): 0, ('H', 'PARENT_RIGHT'): 1,
    ('ArgList', 'NOT'): 0, ('ArgList', 'RESTA'): 0, ('ArgList', 'PARENT_LEFT'): 0, ('ArgList', 'ID'): 0, ('ArgList', 'NUMERO'): 0, ('ArgList', 'STRING'): 0, ('ArgList', 'TRUE'): 0, ('ArgList', 'FALSE'): 0,
    ('ArgList1', 'COMMA'): 0, ('ArgList1', 'PARENT_RIGHT'): 1,
}


def desc_rec_proc(codigo_fuente):
    tokens = lexer_multiples_afds(codigo_fuente) #llamamos al lexer TP1 para que nos devuelva los tokens
    pila = ['EOF', 'Program']
    indice = 0
    forma_sentencial = ['Program']
    derivacion = [forma_sentencial.copy()]
    # Armo un dicionario con los terminales para poder hacer la derivacion de la cadena
    nombres_terminales = {
        'INT': 'int', 'FLOAT': 'float', 'BOOL': 'bool', 'VOID': 'void', 'IF': 'if', 'ELSE': 'else',
        'WHILE': 'while', 'FOR': 'for', 'RETURN': 'return', 'READ': 'read', 'PRINT': 'print',
        'TRUE': 'true', 'FALSE': 'false', 'IGUAL': '==', 'DISTINTO': '!=', 'MENOR_IGUAL': '<=',
        'MAYOR_IGUAL': '>=', 'AND': '&&', 'OR': '||', 'ASIGNACION': '=', 'NOT': '!',
        'MENOR': '<', 'MAYOR': '>', 'SUMA': '+', 'RESTA': '-', 'MULT': '*', 'DIV': '/',
        'PARENT_LEFT': '(', 'PARENT_RIGHT': ')', 'LLAVE_LEFT': '{', 'LLAVE_RIGHT': '}',
        'COMMADOT': ';', 'COMMA': ',', 'STRING': 'str', 'ID': 'id', 'NUMERO': 'num'
    }

    while True:
        if indice >= len(tokens):
            print('La cadena no pertenece al lenguaje: fin de entrada inesperado')
            return False

        tope = pila[-1]
        lookahead = tokens[indice][0]

        if tope == 'EOF' and lookahead == 'EOF':
            break

        if tope in VT:
            if tope == lookahead:
                pila.pop()
                indice += 1
            else:
                print(f'La cadena no pertenece al lenguaje: se esperaba {tope} y se encontró {lookahead}')
                return False
        elif tope in VN:
            indice_produccion = M.get((tope, lookahead))
            if indice_produccion is None:
                print(f'La cadena no pertenece al lenguaje: no hay entrada M[{tope}, {lookahead}]')
                return False

            pila.pop()
            produccion = P[tope][indice_produccion]
            posicion = forma_sentencial.index(tope)
            forma_sentencial = forma_sentencial[:posicion] + produccion + forma_sentencial[posicion + 1:]
            derivacion.append(forma_sentencial.copy())
            pila.extend(reversed(produccion))
        else:
            print(f'La cadena no pertenece al lenguaje: símbolo de pila inesperado {tope}')
            return False

    if indice != len(tokens) - 1:
        print('La cadena no pertenece al lenguaje: quedaron tokens sin consumir')
        return False

    print('La cadena pertenece al lenguaje')
    print('Derivación por la izquierda desde el símbolo distinguido:')
    for paso, forma in enumerate(derivacion):
        forma_legible = ' '.join(nombres_terminales.get(simbolo, simbolo) for simbolo in forma)
        prefijo = '' if paso == 0 else '⇒ '
        print(f'{prefijo}{forma_legible}')
    return True


if __name__ == "__main__":
    with open("codigo10.txt", "r") as f:
        codigo = f.read()
    desc_rec_proc(codigo)

La cadena pertenece al lenguaje
Derivación por la izquierda desde el símbolo distinguido:
Program
⇒ DeclList
⇒ Decl DeclList1
⇒ Type id C DeclList1
⇒ void id C DeclList1
⇒ void id ( B DeclList1
⇒ void id ( ) Block DeclList1
⇒ void id ( ) { StmtList } DeclList1
⇒ void id ( ) { Stmt StmtList1 } DeclList1
⇒ void id ( ) { VarDecl StmtList1 } DeclList1
⇒ void id ( ) { Type id A StmtList1 } DeclList1
⇒ void id ( ) { int id A StmtList1 } DeclList1
⇒ void id ( ) { int id = Expr ; StmtList1 } DeclList1
⇒ void id ( ) { int id = RelExpr Expr1 ; StmtList1 } DeclList1
⇒ void id ( ) { int id = AddExpr E Expr1 ; StmtList1 } DeclList1
⇒ void id ( ) { int id = Term AddExpr1 E Expr1 ; StmtList1 } DeclList1
⇒ void id ( ) { int id = Unary Term1 AddExpr1 E Expr1 ; StmtList1 } DeclList1
⇒ void id ( ) { int id = Factor Term1 AddExpr1 E Expr1 ; StmtList1 } DeclList1
⇒ void id ( ) { int id = num Term1 AddExpr1 E Expr1 ; StmtList1 } DeclList1
⇒ void id ( ) { int id = num AddExpr1 E Expr1 ; StmtList1 } DeclList1